# Stage 0: Install & Config

Install all required packages and define configuration constants.

## Deviations

- **Build Order (§Governance)**: The Google Drive mount and the shared `DRIVE_BASE`, `UNATTENDED_DEFAULT`, and `artifact_gate` definitions are hoisted into the Stage 0 bootstrap (ahead of the pip install and the old Constants cell) so the Drive dependency cache can be consulted before dependencies install. Stage 1's `drive.mount` remains as an idempotent no-op.

In [ ]:
import time, os, sys, subprocess, hashlib, re
_stage0_start = time.time()

# ── Pinned dependencies — single source of truth for BOTH wheel build and
#    every install path, so warm and cold runs yield identical versions (FR-008)
PINNED_REQS = [
    "unsloth",
    "bitsandbytes>=0.43",
    "peft>=0.11",
    "trl>=0.9,<2.0",
    "transformers>=4.46",
    "datasets>=3.0",
    "sentence-transformers>=2.7,<3.0",
    "chromadb>=0.5,<1.0",
    "gradio>=4.0,<5.0",
    "langchain-text-splitters>=0.3",
    "pypdf>=4.0",
]

# ── Bootstrap (DEVIATION §Governance): the Drive mount and the shared
#    UNATTENDED_DEFAULT / artifact_gate definitions are hoisted here — ahead of
#    Stage 1 and the old Constants cell — so the dependency cache can be
#    consulted before pip runs. See the Deviations note at the top of the notebook.
DRIVE_BASE         = "/content/drive/MyDrive/domain-llm-poc"
DEPS_CACHE_DIR     = f"{DRIVE_BASE}/deps_cache"
WHEELS_DIR         = f"{DEPS_CACHE_DIR}/wheels"

UNATTENDED_DEFAULT = None   # None → interactive prompt | "skip" | "rebuild"  (shared with Stage 4/5)
FORCE_REBUILD_DEPS = False  # True → rebuild the wheelhouse next run regardless of validity

_drive_ok = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _drive_ok = True
except Exception as _e:
    print(f"⚠️  Drive not mounted ({_e}); dependency cache disabled for this run.")


def artifact_gate(name, drive_path, sentinel, rebuild_verb="Rebuild"):
    """Decide whether to skip or rebuild a Drive artifact.

    Returns "skip" or "rebuild". Prompts the user only when the artifact
    exists AND UNATTENDED_DEFAULT is None (interactive mode).
    """
    if UNATTENDED_DEFAULT not in (None, "skip", "rebuild"):
        raise ValueError(
            'UNATTENDED_DEFAULT must be None, "skip", or "rebuild" — '
            f"got {UNATTENDED_DEFAULT!r}"
        )
    if not os.path.exists(f"{drive_path}/{sentinel}"):
        return "rebuild"
    if UNATTENDED_DEFAULT is not None:
        return UNATTENDED_DEFAULT
    prompt = (f"{name} found on Drive at {drive_path}.\n"
              f"Skip (s) or {rebuild_verb} (r)?  [default: skip] > ")
    for _attempt in range(2):
        choice = input(prompt).strip().lower()
        if choice in ("", "s", "skip"):
            return "skip"
        if choice in ("r", "rebuild"):
            return "rebuild"
    return "skip"


# ── Dependency-cache helpers ────────────────────────────────────────────────
def runtime_fingerprint():
    """py<major>.<minor>-<cuda tag> (or -cpu). Computable before torch exists."""
    py = f"py{sys.version_info.major}.{sys.version_info.minor}"
    cuda = "cpu"
    try:
        out = subprocess.run(["nvidia-smi"], capture_output=True, text=True,
                             timeout=15).stdout
        m = re.search(r"CUDA Version:\s*([0-9]+)\.([0-9]+)", out)
        if m:
            cuda = f"cu{m.group(1)}{m.group(2)}"
    except Exception:
        cuda = "cpu"
    return f"{py}-{cuda}"


def _manifest_hash(reqs):
    return hashlib.sha256("\n".join(r.strip() for r in reqs).encode()).hexdigest()


def _read(path):
    try:
        with open(path) as f:
            return f.read().strip()
    except Exception:
        return None


def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", *args], check=True)


def _install_online():
    _pip("install", "-q", *PINNED_REQS)


def _rebuild_cache(manifest, fingerprint):
    """Build the wheelhouse from PINNED_REQS, install from it, then persist
    sentinels (manifest written LAST so its presence implies a complete save)."""
    os.makedirs(WHEELS_DIR, exist_ok=True)
    _pip("wheel", "-q", "--wheel-dir", WHEELS_DIR, *PINNED_REQS)
    _pip("install", "-q", "--no-index", "--find-links", WHEELS_DIR, *PINNED_REQS)
    try:
        with open(f"{DEPS_CACHE_DIR}/fingerprint.txt", "w") as f:
            f.write(fingerprint)
        with open(f"{DEPS_CACHE_DIR}/manifest.sha256", "w") as f:
            f.write(manifest)
    except Exception as _e:
        print(f"⚠️  Dependencies installed but cache sentinels not saved ({_e}).")


def _warm_install():
    _pip("install", "-q", "--no-index", "--find-links", WHEELS_DIR, *PINNED_REQS)


# ── Decision + install ──────────────────────────────────────────────────────
_manifest      = _manifest_hash(PINNED_REQS)
_fingerprint   = runtime_fingerprint()
_from_cache    = False

if not _drive_ok:
    print("→ Installing dependencies online (no Drive cache).")
    _install_online()
else:
    _wheels_present = os.path.isdir(WHEELS_DIR) and any(
        fn.endswith(".whl") for fn in os.listdir(WHEELS_DIR))
    _cache_valid = (
        _wheels_present
        and _read(f"{DEPS_CACHE_DIR}/manifest.sha256")  == _manifest      # deps unchanged (FR-004)
        and _read(f"{DEPS_CACHE_DIR}/fingerprint.txt")  == _fingerprint   # runtime match  (FR-005)
    )
    if not _cache_valid:
        _reason = "no usable cache" if not _wheels_present else "dependencies or runtime changed"
        print(f"→ Building dependency cache ({_reason})…")
        try:
            _rebuild_cache(_manifest, _fingerprint)
        except Exception as _e:
            print(f"⚠️  Cache build failed ({_e}); installing online instead.")
            _install_online()
    else:
        _decision = "rebuild" if FORCE_REBUILD_DEPS else artifact_gate(
            "Dependency cache", DEPS_CACHE_DIR, "manifest.sha256", rebuild_verb="Rebuild")
        if _decision == "skip":
            try:
                _warm_install()
                _from_cache = True
            except Exception as _e:
                print(f"⚠️  Offline install failed ({_e}); installing online instead.")
                _install_online()
        else:
            print("→ Rebuilding dependency cache…")
            try:
                _rebuild_cache(_manifest, _fingerprint)
            except Exception as _e:
                print(f"⚠️  Cache rebuild failed ({_e}); installing online instead.")
                _install_online()

_src = "cache (offline)" if _from_cache else "fresh install"
print(f"✅ Stage 0 dependencies ready via {_src} — fingerprint {_fingerprint} "
      f"— {time.time()-_stage0_start:.1f}s")

## Constants

All tunable values defined here — downstream cells reference by name (§III Reproducibility).

In [ ]:
import os, random, shutil, json
import numpy as np
import torch

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_ID        = "unsloth/Qwen3.5-9B"
SEED            = 42
MAX_SEQ_LEN     = 2048

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_R          = 32
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
LORA_TARGETS    = ["q_proj", "k_proj", "v_proj", "o_proj"]

# ── Training ───────────────────────────────────────────────────────────────
TRAIN_EPOCHS    = 3
TRAIN_BATCH     = 4
GRAD_ACCUM      = 2
LEARNING_RATE   = 2e-4

# ── RAG ────────────────────────────────────────────────────────────────────
CHUNK_SIZE      = 2000
CHUNK_OVERLAP   = 200
TOP_K           = 3

# ── Generation ─────────────────────────────────────────────────────────────
MAX_NEW_TOKENS      = 512
TEMPERATURE         = 0.1
TOP_P               = 0.9
REPETITION_PENALTY  = 1.1
ENABLE_THINKING     = False  # MUST be False for both variants (§IV Fair Comparison)

# ── Prompt scaffolding — IDENTICAL across all three panels (§IV) ──────────
SYSTEM_PROMPT      = "You are a helpful assistant. Answer questions clearly and concisely."
RETRIEVAL_TEMPLATE = "Context:\n{context}\n\nQuestion: {question}"

# ── Paths ──────────────────────────────────────────────────────────────────
# DRIVE_BASE is defined in the Stage 0 bootstrap (needed before install).
DOCS_PATH       = "/content/domain_docs"
DATASET_PATH    = "/content/training_dataset.jsonl"


## T4 Fallback Detection

Auto-detects GPU VRAM and switches to the T4 fallback config when < 24 GB.

In [ ]:
_gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
_gpu_name   = torch.cuda.get_device_properties(0).name

if _gpu_mem_gb < 24:
    MODEL_ID       = "unsloth/Qwen3-4B-bnb-4bit"
    MAX_SEQ_LEN    = 512
    LORA_R         = 8
    LORA_ALPHA     = 8
    LORA_TARGETS   = ["q_proj", "v_proj"]
    TRAIN_BATCH    = 1
    GRAD_ACCUM     = 8
    TRAIN_EPOCHS   = 2
    _profile = "T4 fallback config"
else:
    _profile = "A100 config"

print(f"GPU: {_gpu_name} ({_gpu_mem_gb:.1f} GB) → {_profile}")
print(f"MODEL_ID={MODEL_ID}  MAX_SEQ_LEN={MAX_SEQ_LEN}  LORA_R={LORA_R}")


## Seed Fixing

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Seeds fixed: SEED={SEED}")
print(f"Stage 0 complete in {time.time()-_stage0_start:.1f}s")


# Stage 1: Drive Mount

Mount Google Drive and create artifact directories.

In [ ]:
_stage1_start = time.time()
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(f"{DRIVE_BASE}/lora_adapter", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/chroma_index",  exist_ok=True)
print(f"Drive artifact directory: {DRIVE_BASE}")
print(f"Stage 1 complete in {time.time()-_stage1_start:.1f}s")


# Stage 2: Base Model Load

Load the base model in 4-bit NF4 and run a smoke test.

**⚠️ Checkpoint**: Proceed to Stage 3 only after `✅ Base model loaded successfully` appears.

In [ ]:
_stage2_start = time.time()
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = True,
    dtype          = None,
)
print(f"Model loaded: {MODEL_ID}")


In [ ]:
# Smoke test
_smoke_msgs = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": "Hello, what is 2+2?"},
]
_smoke_ids = tokenizer.apply_chat_template(
    _smoke_msgs,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=ENABLE_THINKING,
    return_tensors="pt",
).to("cuda")

with torch.no_grad():
    _smoke_out = model.generate(
        _smoke_ids,
        max_new_tokens     = 64,
        temperature        = TEMPERATURE,
        top_p              = TOP_P,
        do_sample          = True,
        repetition_penalty = REPETITION_PENALTY,
    )

_smoke_new = _smoke_out[0][_smoke_ids.shape[1]:]
_smoke_resp = tokenizer.decode(_smoke_new, skip_special_tokens=True)
print(f"Smoke test response: {_smoke_resp[:200]}")
print(f"✅ Base model loaded successfully")
print(f"Stage 2 complete in {time.time()-_stage2_start:.1f}s")


# Stage 3: Walking Skeleton UI

First shareable demo. Both panels use the base model (no RAG, no adapter) to verify the Gradio plumbing works before adding complexity.

**⚠️ Checkpoint**: Open the share URL, submit a question, verify both panels respond.

In [ ]:
_stage3_start = time.time()

def infer_base(question: str) -> str:
    # No adapter, no RAG — Standard Model panel
    try:
        messages  = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": question},
        ]
        input_ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=ENABLE_THINKING,
            return_tensors="pt",
        ).to("cuda")
        with torch.no_grad():
            out_ids = model.generate(
                input_ids,
                max_new_tokens     = MAX_NEW_TOKENS,
                temperature        = TEMPERATURE,
                top_p              = TOP_P,
                do_sample          = True,
                repetition_penalty = REPETITION_PENALTY,
            )
        new_tokens = out_ids[0][input_ids.shape[1]:]
        return tokenizer.decode(new_tokens, skip_special_tokens=True)
    except Exception:
        return "⚠️ Generation failed — please retry."


In [ ]:
import gradio as gr

with gr.Blocks(title="Domain LLM POC — Walking Skeleton") as skeleton_demo:
    gr.Markdown("## Domain LLM POC — Walking Skeleton (Stage 3)")
    gr.Markdown("_Both panels use the base model. RAG and fine-tuning added in later stages._")
    with gr.Row():
        q_in = gr.Textbox(label="Your question", lines=3, placeholder="Type a question…")
    with gr.Row():
        btn = gr.Button("Submit", variant="primary")
    with gr.Row():
        with gr.Column():
            out_std  = gr.Textbox(label="Standard Model",    interactive=False, lines=10)
        with gr.Column():
            out_spec = gr.Textbox(label="Specialized Model", interactive=False, lines=10)

    btn.click(fn=infer_base, inputs=q_in, outputs=[out_std, out_spec])

skeleton_demo.launch(share=True, server_name="0.0.0.0")
print(f"✅ Walking skeleton UI launched — share URL shown above")
print(f"Stage 3 complete in {time.time()-_stage3_start:.1f}s")


# Stage 4: RAG Pipeline

Load domain documents → chunk → embed → ChromaDB vector index (Drive-backed).

**⚠️ Prerequisite**: Upload domain documents to `/content/domain_docs/` before running.

**Resumability**: If a Drive index already exists, this stage prompts you to skip (load it) or rebuild. Set `UNATTENDED_DEFAULT` to `"skip"` or `"rebuild"` to answer automatically for unattended runs.

In [ ]:
_stage4_start = time.time()
import pypdf
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers    import SentenceTransformer
import chromadb

# Always initialise embed model and ChromaDB client
embed_model   = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path=f"{DRIVE_BASE}/chroma_index")

_gate = artifact_gate("RAG index", f"{DRIVE_BASE}/chroma_index",
                      "chroma.sqlite3", rebuild_verb="Rebuild")
if _gate == "skip":
    collection = chroma_client.get_collection("domain_docs")
    print(f"⏩ Skipping index build — chroma_index loaded from Drive")
    print(f"Loaded existing collection: {collection.count()} chunks")
else:
    # Load documents
    all_texts, all_sources = [], []
    for fpath in sorted(Path(DOCS_PATH).rglob("*")):
        if not fpath.is_file():
            continue
        if fpath.suffix == ".pdf":
            reader = pypdf.PdfReader(str(fpath))
            text   = "\n".join(page.extract_text() or "" for page in reader.pages)
        elif fpath.suffix in (".txt", ".md"):
            text = fpath.read_text(encoding="utf-8", errors="ignore")
        else:
            continue
        if text.strip():
            all_texts.append(text)
            all_sources.append(fpath.name)
    print(f"Loaded {len(all_texts)} documents from {DOCS_PATH}/")

    # Chunk
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
    )
    chunks, metadatas = [], []
    for text, source in zip(all_texts, all_sources):
        for i, chunk in enumerate(splitter.split_text(text)):
            chunks.append(chunk)
            metadatas.append({"source": source, "chunk_id": i})
    print(f"Created {len(chunks)} chunks")

    # Embed
    embeddings = embed_model.encode(
        chunks, batch_size=32, show_progress_bar=True
    ).tolist()

    # Build ChromaDB collection
    try:
        chroma_client.delete_collection("domain_docs")
    except Exception:
        pass
    collection = chroma_client.create_collection(
        "domain_docs", metadata={"hnsw:space": "cosine"}
    )
    ids = [f"chunk_{i}" for i in range(len(chunks))]
    collection.add(
        documents=chunks, embeddings=embeddings,
        metadatas=metadatas, ids=ids,
    )
    print(f"✅ Vector index saved to Drive: {DRIVE_BASE}/chroma_index  ({len(chunks)} chunks)")


In [ ]:
def retrieve(query: str, k: int = TOP_K) -> str:
    # Return top-k relevant chunks as a plain string (raw context, no question appended)
    q_emb    = embed_model.encode([query]).tolist()
    results  = collection.query(query_embeddings=q_emb, n_results=k)
    top_docs = results["documents"][0] if results["documents"] else []
    return "\n---\n".join(top_docs)

print(f"Stage 4 complete in {time.time()-_stage4_start:.1f}s")


# Stage 5: Fine-Tuning

Validate the training JSONL → apply LoRA → SFTTrainer → save adapter to Drive.

**⚠️ Prerequisite**: Upload `training_dataset.jsonl` to `/content/training_dataset.jsonl` before running this stage (minimum 200 pairs).

**Resumability**: If a LoRA adapter already exists on Drive, this stage prompts you to skip (reuse it) or retrain. Set `UNATTENDED_DEFAULT` to `"skip"` or `"rebuild"` to answer automatically for unattended runs.

In [ ]:
_stage5_start = time.time()
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

_gate = artifact_gate("LoRA adapter", f"{DRIVE_BASE}/lora_adapter",
                      "adapter_model.safetensors", rebuild_verb="Retrain")
if _gate == "skip":
    print(f"⏩ Skipping Stage 5 — lora_adapter loaded from Drive")
else:
    # Validate JSONL
    assert os.path.exists(DATASET_PATH), f"Dataset not found: {DATASET_PATH}"
    valid, errors = [], []
    with open(DATASET_PATH, encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            try:
                obj  = json.loads(line)
                msgs = obj.get("messages", [])
                assert len(msgs) >= 2,                   "Need ≥2 messages"
                assert msgs[0]["role"] == "user",        "First role must be user"
                assert msgs[0]["content"].strip(),       "User content is empty"
                assert msgs[1]["role"] == "assistant",   "Second role must be assistant"
                assert msgs[1]["content"].strip(),       "Assistant content is empty"
                valid.append(obj)
            except Exception as e:
                errors.append(f"  Line {i}: {e}")
    if errors:
        raise ValueError("JSONL validation failed:\n" + "\n".join(errors[:10]))
    if len(valid) < 200:
        raise ValueError(f"Need ≥200 training pairs, got {len(valid)}")
    if len(valid) < 500:
        print(f"⚠️  Warning: {len(valid)} pairs (target 500+) — more data may improve results")
    print(f"✅ JSONL valid: {len(valid)} pairs")

    # Backup dataset to Drive
    shutil.copy(DATASET_PATH, f"{DRIVE_BASE}/training_dataset.jsonl")

    # Apply LoRA
    model = FastLanguageModel.get_peft_model(
        model,
        r                          = LORA_R,
        lora_alpha                 = LORA_ALPHA,
        target_modules             = LORA_TARGETS,
        lora_dropout               = LORA_DROPOUT,
        bias                       = "none",
        use_gradient_checkpointing = "unsloth",
    )

    # Load & format dataset
    dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
    def _fmt(ex):
        return {"text": tokenizer.apply_chat_template(
            ex["messages"], tokenize=False, add_generation_prompt=False
        )}
    dataset = dataset.map(_fmt)

    # Train
    trainer = SFTTrainer(
        model              = model,
        tokenizer          = tokenizer,
        train_dataset      = dataset,
        dataset_text_field = "text",
        max_seq_length     = MAX_SEQ_LEN,
        args = TrainingArguments(
            per_device_train_batch_size  = TRAIN_BATCH,
            gradient_accumulation_steps  = GRAD_ACCUM,
            num_train_epochs             = TRAIN_EPOCHS,
            learning_rate                = LEARNING_RATE,
            fp16           = not torch.cuda.is_bf16_supported(),
            bf16           = torch.cuda.is_bf16_supported(),
            logging_steps  = 10,
            output_dir     = "outputs",
            lr_scheduler_type = "cosine",
            seed           = SEED,
        ),
    )
    trainer_stats = trainer.train()
    print(f"Training complete — steps: {trainer_stats.global_step}, "
          f"final loss: {trainer_stats.training_loss:.4f}")

    # Save adapter
    model.save_pretrained(f"{DRIVE_BASE}/lora_adapter")
    tokenizer.save_pretrained(f"{DRIVE_BASE}/lora_adapter")
    print(f"✅ Adapter saved to Drive: {DRIVE_BASE}/lora_adapter")

print(f"Stage 5 complete in {time.time()-_stage5_start:.1f}s")


# Stage 6: Full Pipeline Load

Reload the base model fresh, attach the LoRA adapter, reconnect ChromaDB, and define all three inference functions.

In [ ]:
_stage6_start = time.time()
from unsloth import FastLanguageModel
from peft    import PeftModel
import chromadb
from sentence_transformers import SentenceTransformer

# Reload base model (fresh — discards any in-memory LoRA from Stage 5)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = True,
    dtype          = None,
)

# Attach LoRA adapter
peft_model = PeftModel.from_pretrained(model, f"{DRIVE_BASE}/lora_adapter")
peft_model.eval()
print(f"LoRA adapter loaded from: {DRIVE_BASE}/lora_adapter")

# Reconnect ChromaDB and embedding model
_chroma     = chromadb.PersistentClient(path=f"{DRIVE_BASE}/chroma_index")
collection  = _chroma.get_collection("domain_docs")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"ChromaDB reconnected: {collection.count()} chunks")

def retrieve(query: str, k: int = TOP_K) -> str:
    q_emb   = embed_model.encode([query]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=k)
    return "\n---\n".join(results["documents"][0] if results["documents"] else [])


In [ ]:
def _generate(mdl, messages: list) -> str:
    # Shared helper: tokenize -> generate -> decode
    ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
        return_tensors="pt",
    ).to("cuda")
    with torch.no_grad():
        out = mdl.generate(
            ids,
            max_new_tokens     = MAX_NEW_TOKENS,
            temperature        = TEMPERATURE,
            top_p              = TOP_P,
            do_sample          = True,
            repetition_penalty = REPETITION_PENALTY,
        )
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)


def infer_specialized(question: str) -> str:
    # LoRA adapter + RAG context — Specialized Model panel
    try:
        context  = retrieve(question)
        prompt   = RETRIEVAL_TEMPLATE.format(context=context, question=question)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ]
        return _generate(peft_model, messages)
    except Exception:
        return "⚠️ Generation failed — please retry."


def infer_rag_only(question: str) -> str:
    # Base model + RAG, no adapter — Standard + Docs panel (attribution analysis)
    try:
        context  = retrieve(question)
        prompt   = RETRIEVAL_TEMPLATE.format(context=context, question=question)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ]
        return _generate(model, messages)
    except Exception:
        return "⚠️ Generation failed — please retry."


print("✅ Specialized model ready — infer_base / infer_rag_only / infer_specialized defined")
print(f"Stage 6 complete in {time.time()-_stage6_start:.1f}s")


# Stage 7: Demo UI

Full three-panel comparison UI with live share link.

| Panel | Base model | Adapter | RAG |
|---|---|---|---|
| Standard Model | ✓ | ✗ | ✗ |
| Standard + Docs | ✓ | ✗ | ✓ |
| Specialized Model | ✓ | ✓ | ✓ |

In [ ]:
_stage7_start = time.time()

# Optional: snapshot notebook to Drive for reproducibility (T017)
try:
    shutil.copy("/content/notebook.ipynb", f"{DRIVE_BASE}/notebook.ipynb")
    print(f"✅ Notebook snapshot saved to Drive")
except Exception as e:
    print(f"ℹ️  Notebook snapshot skipped: {e}")


In [ ]:
import gradio as gr

# Close Stage 3 skeleton UI before launching full demo
gr.close_all()


def _run_all(question: str):
    # Run all three panels sequentially; Gradio shows results simultaneously
    return (
        infer_base(question),
        infer_rag_only(question),
        infer_specialized(question),
    )


with gr.Blocks(title="Domain LLM POC — Full Demo") as demo:
    gr.Markdown("## Domain LLM POC — Full Comparison")
    gr.Markdown(
        "| Panel | Model | Adapter | RAG |\n"
        "|---|---|---|---|\n"
        "| Standard Model | Base | ✗ | ✗ |\n"
        "| Standard + Docs | Base | ✗ | ✓ |\n"
        "| Specialized Model | Base | ✓ | ✓ |"
    )
    with gr.Row():
        q_in = gr.Textbox(
            label="Your question", lines=3,
            placeholder="Type a domain question…", scale=4,
        )
    with gr.Row():
        btn = gr.Button("Submit", variant="primary")
    with gr.Row():
        with gr.Column():
            out_std  = gr.Textbox(label="Standard Model",    interactive=False, lines=14)
        with gr.Column():
            out_rag  = gr.Textbox(label="Standard + Docs",   interactive=False, lines=14)
        with gr.Column():
            out_spec = gr.Textbox(label="Specialized Model", interactive=False, lines=14)

    btn.click(
        fn      = _run_all,
        inputs  = q_in,
        outputs = [out_std, out_rag, out_spec],
    )

demo.launch(share=True, server_name="0.0.0.0")
print(f"✅ Full demo UI launched — share URL shown above")
print(f"Stage 7 complete in {time.time()-_stage7_start:.1f}s")


# [Optional] GGUF Export

Export the merged model as a 4-bit GGUF file for local inference.

**Skip if Drive space is limited** (≈5 GB needed). Record in a `## Deviations` section at the top of the notebook if omitted.

In [ ]:
# Run this cell only if you want GGUF for local inference
GGUF_PATH = f"{DRIVE_BASE}/gguf_export"
os.makedirs(GGUF_PATH, exist_ok=True)

model.save_pretrained_merged(
    GGUF_PATH,
    tokenizer,
    save_method="merged_4bit",
)
print(f"✅ GGUF export saved to: {GGUF_PATH}")
